# BITS Pilani - M.Tech (AIML) Machine Learning Assignment 2
## Classification Models & Evaluation on Breast Cancer Wisconsin (Diagnostic) Dataset

This notebook implements and evaluates the 5 required classification models using the exact packages specified in `requirements.txt`:
1. **Logistic Regression**
2. **Decision Tree Classifier**
3. **K-Nearest Neighbor Classifier (KNN)**
4. **Naive Bayes Classifier (Gaussian)**
5. **Support Vector Machine (SVM)**

### Evaluation Metrics:
- Accuracy
- AUC Score
- Precision
- Recall
- F1 Score
- Matthews Correlation Coefficient (MCC Score)

In [5]:
# Step 1: Install required dependencies from requirements.txt
%pip install streamlit==1.50.0 pandas==2.3.3 numpy==2.0.2 scikit-learn==1.6.1 joblib==1.5.3


Note: you may need to restart the kernel to use updated packages.


In [6]:
# Step 2: Import dependencies
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)

print('All libraries imported successfully.')


All libraries imported successfully.


In [7]:
# Step 3: Load Wisconsin Diagnostic Breast Cancer Dataset
dataset = load_breast_cancer(as_frame=True)
FEATURES = list(dataset.feature_names)
LABELS = {0: 'malignant', 1: 'benign'}

print(f'Total instances: {dataset.data.shape[0]} (Requirement: >= 500)')
print(f'Total features: {dataset.data.shape[1]} (Requirement: >= 12)')
print('Target class distribution:')
print(dataset.target.value_counts())


Total instances: 569 (Requirement: >= 500)
Total features: 30 (Requirement: >= 12)
Target class distribution:
target
1    357
0    212
Name: count, dtype: int64


In [8]:
# Step 4: Split into Train (80%) and Test (20%) with stratified sampling
X_train, X_test, y_train, y_test = train_test_split(
    dataset.data, dataset.target, test_size=0.2, random_state=42,
    stratify=dataset.target
)
print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set: {X_test.shape[0]} rows (42 malignant, 72 benign)')


Training set: 455 rows
Test set: 114 rows (42 malignant, 72 benign)


In [9]:
# Step 5: Load pre-trained models from model/ (or train fresh if missing)
BASE_DIR = Path('.')
MODEL_FILES = {
    'Logistic Regression': 'model/logistic_regression.joblib',
    'Decision Tree': 'model/decision_tree.joblib',
    'KNN': 'model/knn.joblib',
    'Gaussian Naive Bayes': 'model/gaussian_naive_bayes.joblib',
    'Support Vector Machine': 'model/support_vector_machine.joblib',
}

models = {}
for name, rel_path in MODEL_FILES.items():
    p = BASE_DIR / rel_path
    if p.is_file():
        models[name] = joblib.load(p)
        print(f'Loaded saved model: {name}')
    else:
        print(f'Model file not found for {name}, training fresh...')

if len(models) < len(MODEL_FILES):
    fresh_definitions = {
        'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42)),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier()),
        'Gaussian Naive Bayes': GaussianNB(),
        'Support Vector Machine': make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
    }
    for name, model_def in fresh_definitions.items():
        if name not in models:
            model_def.fit(X_train, y_train)
            models[name] = model_def
            print(f'Trained model: {name}')


Loaded saved model: Logistic Regression
Loaded saved model: Decision Tree
Loaded saved model: KNN
Loaded saved model: Gaussian Naive Bayes
Loaded saved model: Support Vector Machine


In [10]:
# Step 6: Evaluate all 5 models on test_data.csv and compute all 6 metrics
test_csv_path = BASE_DIR / 'test_data.csv'
if test_csv_path.is_file():
    test_df = pd.read_csv(test_csv_path)
    eval_X = test_df[FEATURES].apply(pd.to_numeric)
    eval_y = test_df['target'].astype(int)
    print(f'Evaluating on {test_csv_path} ({len(test_df)} rows)')
else:
    eval_X, eval_y = X_test, y_test
    print(f'Evaluating on held-out test split ({len(eval_y)} rows)')

rows = []
for name, model in models.items():
    pred = model.predict(eval_X)
    pos_col = list(model.classes_).index(1)
    prob = model.predict_proba(eval_X)[:, pos_col]
    
    rows.append({
        'ML Model Name': name,
        'Accuracy': accuracy_score(eval_y, pred),
        'AUC': roc_auc_score(eval_y, prob),
        'Precision': precision_score(eval_y, pred, zero_division=0),
        'Recall': recall_score(eval_y, pred, zero_division=0),
        'F1': f1_score(eval_y, pred, zero_division=0),
        'MCC': matthews_corrcoef(eval_y, pred),
    })

metrics_df = pd.DataFrame(rows)
print('\n=== COMPARISON TABLE: EVALUATION METRICS FOR ALL 5 MODELS ===')
display_df = metrics_df.copy()
for col in ['Accuracy', 'AUC', 'Precision', 'Recall', 'F1', 'MCC']:
    display_df[col] = display_df[col].map(lambda v: f'{v:.4f}')
print(display_df.to_string(index=False))


Evaluating on test_data.csv (114 rows)

=== COMPARISON TABLE: EVALUATION METRICS FOR ALL 5 MODELS ===
         ML Model Name Accuracy    AUC Precision Recall     F1    MCC
   Logistic Regression   0.9825 0.9954    0.9861 0.9861 0.9861 0.9623
         Decision Tree   0.9123 0.9157    0.9559 0.9028 0.9286 0.8174
                   KNN   0.9561 0.9788    0.9589 0.9722 0.9655 0.9054
  Gaussian Naive Bayes   0.9386 0.9878    0.9452 0.9583 0.9517 0.8676
Support Vector Machine   0.9825 0.9950    0.9861 0.9861 0.9861 0.9623


In [11]:
# Step 7: Confusion Matrix & Classification Report for each of the 5 models
for name, model in models.items():
    pred = model.predict(eval_X)
    print('=' * 65)
    print(f'MODEL: {name}')
    print('=' * 65)
    cm = confusion_matrix(eval_y, pred, labels=[0, 1])
    cm_df = pd.DataFrame(
        cm,
        index=['Actual Malignant (0)', 'Actual Benign (1)'],
        columns=['Predicted Malignant (0)', 'Predicted Benign (1)']
    )
    print('\nConfusion Matrix:')
    print(cm_df)
    print('\nClassification Report:')
    print(classification_report(
        eval_y, pred, labels=[0, 1],
        target_names=['Malignant (0)', 'Benign (1)'],
        zero_division=0
    ))


MODEL: Logistic Regression

Confusion Matrix:
                      Predicted Malignant (0)  Predicted Benign (1)
Actual Malignant (0)                       41                     1
Actual Benign (1)                           1                    71

Classification Report:
               precision    recall  f1-score   support

Malignant (0)       0.98      0.98      0.98        42
   Benign (1)       0.99      0.99      0.99        72

     accuracy                           0.98       114
    macro avg       0.98      0.98      0.98       114
 weighted avg       0.98      0.98      0.98       114

MODEL: Decision Tree

Confusion Matrix:
                      Predicted Malignant (0)  Predicted Benign (1)
Actual Malignant (0)                       39                     3
Actual Benign (1)                           7                    65

Classification Report:
               precision    recall  f1-score   support

Malignant (0)       0.85      0.93      0.89        42
   Benign (1) 